# 02 — Sentimento das manchetes com FinBERT

Material de apoio ao [artigo](00_artigo.ipynb). Aplicamos `ProsusAI/finbert`, sem treinamento, às notícias limpas do notebook 01. Salvamos probabilidades por manchete, o agregado por empresa/data civil e uma revisão qualitativa de 50 casos. Execute as células em ordem; o primeiro carregamento precisa dos pesos locais ou de acesso ao Hugging Face.

## 1. Configuração

Fixamos a revisão dos pesos e do tokenizador. `SOMENTE_CACHE=False` permite o primeiro download; `REPROCESSAR=False` reutiliza probabilidades apenas com configuração compatível. A inferência usa CPU e lotes de 16 textos. Os arquivos de entrada permanecem intactos.

In [1]:
import os
os.environ['MKL_THREADING_LAYER'] = 'SEQUENTIAL'
import sys
import json
import hashlib
import time
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from IPython.display import display

RAIZ = Path.cwd().resolve()
if RAIZ.name == 'notebooks':
    RAIZ = RAIZ.parent
ENTRADA = RAIZ / 'data/processed/noticias_limpas.csv'
if not ENTRADA.is_file():
    raise FileNotFoundError('Abra este notebook na pasta do projeto ou em notebooks/. Falta data/processed/noticias_limpas.csv.')
SAIDA = RAIZ / 'data/processed'
DOCS = RAIZ / 'docs'
for pasta in [SAIDA, DOCS, DOCS / 'revisao_manual', RAIZ / 'data/cache']:
    pasta.mkdir(parents=True, exist_ok=True)

MODELO = 'ProsusAI/finbert'
REVISAO = '4556d13015211d73dccd3fdd39d39232506f3e43'
SOMENTE_CACHE = False
REPROCESSAR = False
TAMANHO_LOTE = 16
MAX_TOKENS = 512
TOL_EMPATE = 1e-8
SEMENTE = 20261015
DISPOSITIVO = torch.device('cpu')
torch.set_num_threads(min(4, os.cpu_count() or 1))
VERSOES = {p: version(p) for p in ['pandas', 'numpy', 'torch', 'transformers', 'tokenizers', 'huggingface-hub']}

def sha256_arquivo(caminho):
    h = hashlib.sha256()
    with Path(caminho).open('rb') as f:
        for bloco in iter(lambda: f.read(1024 * 1024), b''):
            h.update(bloco)
    return h.hexdigest()

HASH_ENTRADA = sha256_arquivo(ENTRADA)
INICIO_EXECUCAO = datetime.now(timezone.utc).isoformat()
print('Python:', sys.version.split()[0], '| ambiente:', Path(sys.executable).parent.name)
print('Entrada:', ENTRADA.relative_to(RAIZ))
print('Dispositivo:', DISPOSITIVO, '| revisão:', REVISAO)
display(pd.Series(VERSOES, name='versao').to_frame())

Python: 3.11.16 | ambiente: aed-sentimento
Entrada: data\processed\noticias_limpas.csv
Dispositivo: cpu | revisão: 4556d13015211d73dccd3fdd39d39232506f3e43


,versao
pandas,3.0.6
numpy,2.4.6
torch,2.14.1+cpu
transformers,5.19.0
tokenizers,0.23.3
huggingface-hub,2.2.0


## 2. Conferir notícias e identificar textos distintos

Verificamos campos, IDs, datas e a chave empresa/data/manchete. O hash identifica o texto para cache e junção. Classificar uma vez cada texto distinto economiza processamento sem eliminar associações entre empresas. Na referência, 6.552 associações correspondem a 6.019 textos.

In [2]:
noticias = pd.read_csv(ENTRADA, parse_dates=['news_date'], keep_default_na=False)
obrigatorias = {'source_id', 'stock', 'news_date', 'headline', 'compartilhada_entre_empresas'}
assert obrigatorias.issubset(noticias.columns), obrigatorias - set(noticias.columns)
assert noticias['source_id'].is_unique
assert noticias['news_date'].notna().all()
assert noticias['headline'].astype(str).str.strip().ne('').all()
assert not noticias.duplicated(['stock', 'news_date', 'headline']).any()
COLUNAS_ORIGINAIS = noticias.columns.tolist()
noticias['headline_hash'] = noticias['headline'].map(lambda s: hashlib.sha256(s.encode('utf-8')).hexdigest())
unicas = noticias[['headline_hash', 'headline']].drop_duplicates().copy()
assert unicas['headline_hash'].is_unique, 'Verificar colisão de hash ou texto inconsistente.'
print(f'Associações manchete–empresa: {len(noticias):,}')
print(f'Textos distintos para inferência: {len(unicas):,}')
print('Dias com notícias:', noticias.groupby(['stock', 'news_date']).ngroups)
display(noticias.groupby('stock').size().rename('n_noticias').to_frame())

Associações manchete–empresa: 6,552
Textos distintos para inferência: 6,019
Dias com notícias: 2569


,n_noticias
stock,
BAC,871
JNJ,1046
KO,644
NFLX,2344
NVDA,1647


## 3. Carregar o modelo pronto

O tokenizador representa os textos numericamente. O modelo em modo `eval()` fornece logits; a softmax produz as três probabilidades. Consultamos `id2label` para associar corretamente cada posição à classe, sem presumir sua ordem.

In [3]:
parametros_carga = dict(revision=REVISAO, local_files_only=SOMENTE_CACHE)
try:
    tokenizador = AutoTokenizer.from_pretrained(MODELO, **parametros_carga)
    modelo = AutoModelForSequenceClassification.from_pretrained(
        MODELO, use_safetensors=False, **parametros_carga
    ).to(DISPOSITIVO).eval()
except OSError as exc:
    raise RuntimeError('Não foi possível carregar a revisão fixada do modelo. Se SOMENTE_CACHE=True, use False para permitir o primeiro download; caso contrário, verifique a conexão e o acesso ao Hugging Face.') from exc

rotulos = {int(i): str(nome).lower() for i, nome in modelo.config.id2label.items()}
assert set(rotulos.values()) == {'positive', 'negative', 'neutral'}, rotulos
INDICES = {nome: i for i, nome in rotulos.items()}
TRADUCAO = {'positive': 'positivo', 'negative': 'negativo', 'neutral': 'neutro'}
PROB_COLS = ['p_positivo', 'p_negativo', 'p_neutro']
CLASSES = np.array(['positivo', 'negativo', 'neutro'])
print('Posições fornecidas pelo modelo:', rotulos)
print('Modelo carregado em modo de inferência.')

Posições fornecidas pelo modelo: {0: 'positive', 1: 'negative', 2: 'neutral'}
Modelo carregado em modo de inferência.


Loading weights: 100%|##########| 201/201 [00:00<00:00, 20790.00it/s]


## 4. Definir as medidas e conferir o funcionamento

O escore é `P(positivo) − P(negativo)`. A classe usa a maior probabilidade, com empate explícito à tolerância de `1e-8`. Escore próximo de zero pode resultar de tons opostos; não define, sozinho, neutralidade. A maior probabilidade não é uma estimativa calibrada de acerto. Três frases sintéticas verificam o funcionamento básico, sem avaliar precisão na base real.

In [4]:
def validar_probabilidades(tabela):
    p = tabela[PROB_COLS].to_numpy(dtype=float)
    assert np.isfinite(p).all()
    assert ((p >= 0) & (p <= 1)).all()
    assert np.allclose(p.sum(axis=1), 1, atol=1e-5, rtol=0)

def acrescentar_medidas(tabela, coluna_classe):
    tabela = tabela.copy()
    validar_probabilidades(tabela)
    p = tabela[PROB_COLS].to_numpy(dtype=float)
    maiores = p.max(axis=1)
    empates = np.isclose(p, maiores[:, None], atol=TOL_EMPATE, rtol=0).sum(axis=1) > 1
    tabela[coluna_classe] = np.where(empates, 'empate', CLASSES[p.argmax(axis=1)])
    tabela['escore_sentimento'] = tabela['p_positivo'] - tabela['p_negativo']
    tabela['probabilidade_maxima'] = maiores
    tabela['margem_top2'] = np.diff(np.sort(p, axis=1)[:, -2:], axis=1).ravel()
    tabela['empate_probabilidades'] = empates
    assert tabela['escore_sentimento'].between(-1, 1).all()
    return tabela

def classificar_textos(textos):
    entradas = tokenizador(textos, padding=True, truncation=True, max_length=MAX_TOKENS, return_tensors='pt')
    entradas = {k: v.to(DISPOSITIVO) for k, v in entradas.items()}
    with torch.inference_mode():
        probabilidades = torch.softmax(modelo(**entradas).logits, dim=-1).cpu().numpy()
    tabela = pd.DataFrame({
        'headline': textos,
        'p_positivo': probabilidades[:, INDICES['positive']],
        'p_negativo': probabilidades[:, INDICES['negative']],
        'p_neutro': probabilidades[:, INDICES['neutral']],
    })
    return acrescentar_medidas(tabela, 'classe_finbert')

exemplos = [
    'The company reported record profits and strong revenue growth.',
    'The company reported losses and cut its earnings forecast.',
    'The company will publish its quarterly report on Monday.',
]
teste = classificar_textos(exemplos)
display(teste[['headline', *PROB_COLS, 'classe_finbert', 'escore_sentimento']])
assert teste['classe_finbert'].tolist() == ['positivo', 'negativo', 'neutro'], 'Investigue o teste básico antes de continuar.'

,headline,p_positivo,p_negativo,p_neutro,classe_finbert,escore_sentimento
0,The company reported record profits and strong...,0.952856,0.014632,0.032511,positivo,0.938224
1,The company reported losses and cut its earnin...,0.008190,0.970416,0.021394,negativo,-0.962227
2,The company will publish its quarterly report ...,0.017976,0.055278,0.926746,neutro,-0.037302


## 5. Conferir o comprimento das manchetes

Contamos tokens antes de limitar os textos a 512. Eventuais cortes são sinalizados e incluídos na seleção para revisão. Agrupar textos de comprimentos semelhantes reduz preenchimento desnecessário nos lotes. O modelo classifica a manchete inteira, sem distinguir o tom para cada empresa citada.

In [5]:
tokens_sem_corte = tokenizador(unicas['headline'].tolist(), truncation=False, padding=False, return_length=True)
unicas['n_tokens'] = tokens_sem_corte['length']
unicas['foi_truncada'] = unicas['n_tokens'].gt(MAX_TOKENS)
print('Textos com mais de 512 tokens:', int(unicas['foi_truncada'].sum()))
display(unicas['n_tokens'].describe().to_frame('tokens_por_texto'))

Textos com mais de 512 tokens: 0


,tokens_por_texto
count,6019.000000
mean,22.196378
std,12.271805
min,6.000000
25%,14.000000
50%,18.000000
75%,25.000000
max,91.000000


## 6. Classificar e salvar o cache

O cache combina hash do texto, revisão, parâmetros e versões. Reutilizamos apenas resultados compatíveis e salvamos o progresso a cada 16 lotes, permitindo retomar uma execução interrompida. O arquivo temporário substitui o anterior depois da gravação completa.

In [6]:
config_cache = {
    'modelo': MODELO, 'revisao_modelo': REVISAO, 'revisao_tokenizador': REVISAO,
    'max_tokens': MAX_TOKENS, 'truncation': True, 'dtype': 'float32',
    'transformers': VERSOES['transformers'], 'torch': VERSOES['torch'], 'tokenizers': VERSOES['tokenizers'],
}
CHAVE_CACHE = hashlib.sha256(json.dumps(config_cache, sort_keys=True).encode()).hexdigest()
PASTA_CACHE = RAIZ / 'data/cache' / CHAVE_CACHE[:16]
PASTA_CACHE.mkdir(parents=True, exist_ok=True)
ARQ_CACHE = PASTA_CACHE / 'probabilidades.csv'
ARQ_CONFIG = PASTA_CACHE / 'configuracao.json'
COL_CACHE = ['headline_hash', 'headline', *PROB_COLS]
cache = pd.DataFrame(columns=COL_CACHE)
if ARQ_CACHE.exists() and not REPROCESSAR:
    assert ARQ_CONFIG.is_file(), 'Cache sem configuração; investigue antes de reutilizar.'
    assert json.loads(ARQ_CONFIG.read_text(encoding='utf-8')) == config_cache
    cache = pd.read_csv(ARQ_CACHE, keep_default_na=False)
    assert cache['headline_hash'].is_unique
    assert cache['headline_hash'].equals(cache['headline'].map(lambda s: hashlib.sha256(s.encode('utf-8')).hexdigest()))
    validar_probabilidades(cache)
ARQ_CONFIG.write_text(json.dumps(config_cache, indent=2, ensure_ascii=False), encoding='utf-8')
cache = cache.loc[cache['headline_hash'].isin(unicas['headline_hash']), COL_CACHE].copy()
pendentes = unicas.loc[~unicas['headline_hash'].isin(cache['headline_hash'])].sort_values(['n_tokens', 'headline_hash'])
registros = cache.to_dict('records')

def salvar_cache():
    tabela = pd.DataFrame(registros, columns=COL_CACHE).sort_values('headline_hash').reset_index(drop=True)
    assert tabela['headline_hash'].is_unique
    validar_probabilidades(tabela)
    temporario = ARQ_CACHE.with_suffix('.tmp')
    tabela.to_csv(temporario, index=False, float_format='%.10g')
    temporario.replace(ARQ_CACHE)

print(f'Textos no cache: {len(cache):,} | a processar: {len(pendentes):,}', flush=True)
inicio = time.perf_counter()
for numero_lote, inicio_lote in enumerate(range(0, len(pendentes), TAMANHO_LOTE), start=1):
    lote = pendentes.iloc[inicio_lote:inicio_lote + TAMANHO_LOTE]
    resultado_lote = classificar_textos(lote['headline'].tolist())
    resultado_lote['headline_hash'] = lote['headline_hash'].to_numpy()
    registros.extend(resultado_lote[COL_CACHE].to_dict('records'))
    if numero_lote % 16 == 0 or inicio_lote + TAMANHO_LOTE >= len(pendentes):
        salvar_cache()
        print(f'{len(registros):,}/{len(unicas):,} textos salvos | {time.perf_counter() - inicio:.1f} s', flush=True)
if not ARQ_CACHE.exists():
    salvar_cache()
probabilidades = pd.read_csv(ARQ_CACHE, keep_default_na=False)
probabilidades = probabilidades.loc[probabilidades['headline_hash'].isin(unicas['headline_hash'])].reset_index(drop=True)
assert set(probabilidades['headline_hash']) == set(unicas['headline_hash'])
assert probabilidades['headline_hash'].is_unique
validar_probabilidades(probabilidades)
print('Inferência/cache concluídos. Todas as manchetes possuem probabilidades.')

Textos no cache: 6,019 | a processar: 0
Inferência/cache concluídos. Todas as manchetes possuem probabilidades.


## 7. Reassociar probabilidades e exportar

Uma junção `many_to_one` devolve a classificação às associações originais. Conferimos que nenhum ID, texto, empresa ou data foi alterado, perdido ou multiplicado. Salvamos probabilidades, classe, escore e indicadores de truncamento e relemos a saída.

In [7]:
medidas = acrescentar_medidas(probabilidades, 'classe_finbert')
medidas = medidas.drop(columns='headline').merge(
    unicas[['headline_hash', 'n_tokens', 'foi_truncada']], on='headline_hash', validate='one_to_one'
)
classificadas = noticias.merge(medidas, on='headline_hash', how='left', validate='many_to_one', sort=False)
assert len(classificadas) == len(noticias)
assert classificadas['source_id'].is_unique
pd.testing.assert_frame_equal(classificadas[COLUNAS_ORIGINAIS], noticias[COLUNAS_ORIGINAIS])
assert classificadas[PROB_COLS].notna().all().all()
validar_probabilidades(classificadas)
classificadas['modelo_finbert'] = MODELO
classificadas['revisao_finbert'] = REVISAO
ARQ_CLASSIFICADAS = SAIDA / 'noticias_com_sentimento.csv'
classificadas.to_csv(ARQ_CLASSIFICADAS, index=False, date_format='%Y-%m-%d', float_format='%.10g')
relidas = pd.read_csv(ARQ_CLASSIFICADAS, parse_dates=['news_date'], keep_default_na=False)
assert len(relidas) == len(noticias) and relidas['source_id'].is_unique
validar_probabilidades(relidas)
assert sha256_arquivo(ENTRADA) == HASH_ENTRADA, 'A entrada foi alterada.'
display(classificadas['classe_finbert'].value_counts().rename_axis('classe').to_frame('associacoes'))
print('Tabela por notícia salva:', ARQ_CLASSIFICADAS.name)

Tabela por notícia salva: noticias_com_sentimento.csv


,associacoes
classe,
neutro,3598
positivo,1613
negativo,1341


## 8. Agregar por empresa e data civil

Calculamos médias das probabilidades e a contagem de manchetes. A classe diária usa a maior probabilidade média, com empates explícitos. Somente datas com notícias aparecem; ausência não recebe classe neutra. Esse agregado serve à evolução temporal. O notebook 03 reagrupará as manchetes depois de associá-las ao pregão posterior.

In [8]:
diario = classificadas.groupby(['stock', 'news_date'], as_index=False).agg(
    p_positivo=('p_positivo', 'mean'), p_negativo=('p_negativo', 'mean'), p_neutro=('p_neutro', 'mean'),
    n_noticias=('source_id', 'size'), n_compartilhadas=('compartilhada_entre_empresas', 'sum'),
    n_truncadas=('foi_truncada', 'sum'), escore_medio_conferencia=('escore_sentimento', 'mean')
)
diario = acrescentar_medidas(diario, 'classe_diaria')
assert np.allclose(diario['escore_sentimento'], diario['escore_medio_conferencia'], atol=1e-8, rtol=0)
diario = diario.drop(columns='escore_medio_conferencia')
assert not diario.duplicated(['stock', 'news_date']).any()
assert diario['n_noticias'].sum() == len(classificadas)
assert diario['n_noticias'].ge(1).all()
ARQ_DIARIO = SAIDA / 'sentimento_diario_noticias.csv'
diario.to_csv(ARQ_DIARIO, index=False, date_format='%Y-%m-%d', float_format='%.10g')
diario_relido = pd.read_csv(ARQ_DIARIO, parse_dates=['news_date'])
validar_probabilidades(diario_relido)
assert diario_relido['n_noticias'].sum() == len(classificadas)
print('Pares empresa/data:', len(diario))
display(diario.head(10))

Pares empresa/data: 2569


,stock,news_date,p_positivo,p_negativo,p_neutro,n_noticias,n_compartilhadas,n_truncadas,classe_diaria,escore_sentimento,probabilidade_maxima,margem_top2,empate_probabilidades
0,BAC,2017-01-04,0.022399,0.760942,0.216659,1,1,0,negativo,-0.738544,0.760942,0.544283,False
1,BAC,2017-01-05,0.339192,0.059916,0.600892,1,0,0,neutro,0.279277,0.600892,0.261699,False
2,BAC,2017-01-09,0.485699,0.085394,0.428907,3,0,0,positivo,0.400304,0.485699,0.056791,False
3,BAC,2017-01-10,0.414023,0.540677,0.045300,2,0,0,negativo,-0.126653,0.540677,0.126653,False
4,BAC,2017-01-11,0.234707,0.235747,0.529546,5,0,0,neutro,-0.001039,0.529546,0.293799,False
5,BAC,2017-01-12,0.266264,0.136560,0.597176,6,0,0,neutro,0.129704,0.597176,0.330911,False
6,BAC,2017-01-13,0.346500,0.114503,0.538998,9,4,0,neutro,0.231997,0.538998,0.192498,False
7,BAC,2017-01-17,0.102495,0.357466,0.540039,3,1,0,neutro,-0.254971,0.540039,0.182573,False
8,BAC,2017-01-18,0.037093,0.044431,0.918477,2,0,0,neutro,-0.007338,0.918477,0.874046,False
9,BAC,2017-01-23,0.050492,0.020471,0.929037,1,1,0,neutro,0.030022,0.929037,0.878545,False


## 9. Revisão humana de 50 manchetes

A amostra dirigida cobre empresas/classes, baixa margem, compartilhamento e possíveis truncamentos; a complementação é aleatória com semente fixa. Edite apenas `rotulo_humano`, `observacao` e `revisor`. São aceitos positivo, negativo, neutro e ambiguo. O arquivo já preenchido é preservado e validado contra os dados originais.

Na revisão existente, `caio` registrou 39 concordâncias e 11 divergências. A seleção dirigida, um único revisor e a exposição às previsões impedem usar 39/50 como acurácia geral. As anotações não substituem as classes automáticas.

In [9]:
def ler_revisao(caminho):
    tabela = pd.read_csv(caminho, sep=None, engine='python', encoding='utf-8-sig',
                         keep_default_na=False, dtype=str)
    exigidas = {'source_id', 'stock', 'headline', 'classe_finbert', 'rotulo_humano', 'observacao', 'revisor'}
    assert exigidas.issubset(tabela.columns), 'A revisão precisa manter as colunas originais e as três colunas humanas.'
    assert len(tabela) == min(50, len(unicas)), 'Preserve todas as linhas da amostra e preencha apenas as colunas humanas.'
    assert tabela['source_id'].is_unique, 'Há IDs repetidos na revisão.'
    origem = classificadas.assign(source_id=classificadas['source_id'].astype(str)).set_index('source_id')
    assert tabela['source_id'].isin(origem.index).all(), 'Há IDs desconhecidos na revisão.'
    for coluna in ['headline', 'stock', 'classe_finbert']:
        assert tabela[coluna].eq(tabela['source_id'].map(origem[coluna])).all(), f'Campo original alterado na revisão: {coluna}'
    return tabela

selecionadas = []
vistos = set()
def incluir_revisao(candidatas, quantidade, motivo):
    if quantidade <= 0 or len(selecionadas) >= 50:
        return
    adicionadas = 0
    for _, linha in candidatas.iterrows():
        if linha['headline_hash'] in vistos:
            continue
        registro = linha.to_dict()
        registro['motivo_selecao'] = motivo
        selecionadas.append(registro)
        vistos.add(linha['headline_hash'])
        adicionadas += 1
        if adicionadas >= quantidade or len(selecionadas) >= 50:
            break

for _, grupo in classificadas.groupby(['stock', 'classe_finbert'], sort=True):
    incluir_revisao(grupo.sample(frac=1, random_state=SEMENTE), 2, 'cobertura_empresa_classe')
incluir_revisao(classificadas.sort_values(['margem_top2', 'source_id']), 10, 'baixa_margem')
incluir_revisao(classificadas.loc[classificadas['compartilhada_entre_empresas']].sample(frac=1, random_state=SEMENTE), 5, 'compartilhada')
truncadas = classificadas.loc[classificadas['foi_truncada']].sort_values('source_id')
if len(truncadas):
    incluir_revisao(truncadas, min(5, 50 - len(selecionadas)), 'truncada')
if len(selecionadas) < 50:
    incluir_revisao(classificadas.sample(frac=1, random_state=SEMENTE), 50 - len(selecionadas), 'aleatoria_complementar')
amostra = pd.DataFrame(selecionadas).head(50)
assert len(amostra) == min(50, len(unicas)) and amostra['headline_hash'].is_unique
empresas_texto = classificadas.groupby('headline_hash')['stock'].agg(lambda x: ', '.join(sorted(set(x))))
amostra['empresas_associadas'] = amostra['headline_hash'].map(empresas_texto)
amostra = amostra[['source_id', 'stock', 'news_date', 'empresas_associadas', 'headline', *PROB_COLS,
                   'classe_finbert', 'escore_sentimento', 'margem_top2', 'foi_truncada', 'motivo_selecao']].copy()
for coluna in ['rotulo_humano', 'observacao', 'revisor']:
    amostra[coluna] = ''
ARQ_REVISAO = DOCS / 'revisao_manual' / f'amostra_50_{HASH_ENTRADA[:8]}_{CHAVE_CACHE[:8]}.csv'
if ARQ_REVISAO.exists():
    print('Arquivo de revisão existente preservado:', ARQ_REVISAO.name)
else:
    amostra.to_csv(ARQ_REVISAO, index=False, date_format='%Y-%m-%d', encoding='utf-8-sig', float_format='%.10g')
revisao_salva = ler_revisao(ARQ_REVISAO)
assert len(revisao_salva) == min(50, len(unicas)), 'O número de linhas da revisão foi alterado; preserve as linhas e preencha apenas as colunas humanas.'
assert revisao_salva['source_id'].is_unique and revisao_salva['source_id'].isin(noticias['source_id'].astype(str)).all()
rotulos_humanos = revisao_salva['rotulo_humano'].astype(str).str.strip().str.lower()
ROTULOS_VALIDOS = {'positivo', 'negativo', 'neutro', 'ambiguo'}
invalidos = rotulos_humanos.ne('') & ~rotulos_humanos.isin(ROTULOS_VALIDOS)
assert not invalidos.any(), 'Há rótulos humanos inválidos no CSV. Use positivo, negativo, neutro ou ambiguo.'
N_REVISAO_VALIDOS = int(rotulos_humanos.isin(ROTULOS_VALIDOS).sum())
ESTADO_REVISAO = 'preenchida' if N_REVISAO_VALIDOS == len(revisao_salva) else ('parcial' if N_REVISAO_VALIDOS else 'pendente')
print(f'Revisão humana: {ESTADO_REVISAO} ({N_REVISAO_VALIDOS}/{len(revisao_salva)} rótulos válidos).')
print('Preencher o arquivo não constitui avaliação formal de precisão.')
display(revisao_salva[['stock', 'headline', 'classe_finbert', 'motivo_selecao', 'rotulo_humano']].head(8))

# Usamos as classes automáticas originais; os rótulos humanos não as substituem.
comparacao_revisao = revisao_salva.copy()
comparacao_revisao['rotulo_humano_normalizado'] = rotulos_humanos
classes_originais = classificadas.set_index(classificadas['source_id'].astype(str))['classe_finbert']
comparacao_revisao['classe_finbert_original'] = comparacao_revisao['source_id'].map(classes_originais)
CLASSES_RECONHECIDAS = ['positivo', 'neutro', 'negativo']
comparaveis = comparacao_revisao['rotulo_humano_normalizado'].isin(CLASSES_RECONHECIDAS) & comparacao_revisao['classe_finbert_original'].isin(CLASSES_RECONHECIDAS)
concordantes = comparacao_revisao['rotulo_humano_normalizado'].eq(comparacao_revisao['classe_finbert_original'])
N_COMPARAVEIS = int(comparaveis.sum())
N_CONCORDANCIAS = int((comparaveis & concordantes).sum())
N_DIVERGENCIAS = int((comparaveis & ~concordantes).sum())
N_AMBIGUOS = int(rotulos_humanos.eq('ambiguo').sum())
matriz_revisao = pd.crosstab(
    comparacao_revisao.loc[comparaveis, 'rotulo_humano_normalizado'],
    comparacao_revisao.loc[comparaveis, 'classe_finbert_original']
).reindex(index=CLASSES_RECONHECIDAS, columns=CLASSES_RECONHECIDAS, fill_value=0)
matriz_revisao.index.name = 'rotulo_humano'
matriz_revisao.columns.name = 'classe_modelo'
print(f'Comparáveis: {N_COMPARAVEIS} | concordâncias: {N_CONCORDANCIAS} | divergências: {N_DIVERGENCIAS} | ambíguos: {N_AMBIGUOS}')
print('A amostra é dirigida; essas contagens não estimam a acurácia geral do modelo.')
display(matriz_revisao)
if N_DIVERGENCIAS:
    display(comparacao_revisao.loc[comparaveis & ~concordantes,
        ['source_id', 'headline', 'classe_finbert_original', 'rotulo_humano', 'observacao']])


Arquivo de revisão existente preservado: amostra_50_cf0a9799_e1f584a3.csv
Revisão humana: preenchida (50/50 rótulos válidos).
Preencher o arquivo não constitui avaliação formal de precisão.
Comparáveis: 50 | concordâncias: 39 | divergências: 11 | ambíguos: 0
A amostra é dirigida; essas contagens não estimam a acurácia geral do modelo.


,stock,headline,classe_finbert,motivo_selecao,rotulo_humano
0,BAC,Shares of several financial companies are trad...,negativo,cobertura_empresa_classe,negativo
1,BAC,"Benzinga's Top Upgrades, Downgrades For August...",negativo,cobertura_empresa_classe,neutro
2,BAC,"The Market In 5 Minutes: Big Bank Earnings, GM...",neutro,cobertura_empresa_classe,neutro
3,BAC,Memo Shows Merrill Lynch to Cut Divisions to 6...,neutro,cobertura_empresa_classe,positivo
4,BAC,Stock Market Gets Lift From Earnings As Geopol...,positivo,cobertura_empresa_classe,positivo
5,BAC,13F For Buffett's Berkshire Shows Raised Stake...,positivo,cobertura_empresa_classe,positivo
6,JNJ,The Daily Biotech Pulse: AEterna Zentaris Expl...,negativo,cobertura_empresa_classe,neutro
7,JNJ,Johnson & Johnson Shares Down 1.64% Premarket,negativo,cobertura_empresa_classe,negativo


classe_modelo,positivo,neutro,negativo
rotulo_humano,,,
positivo,11,3,0
neutro,2,20,6
negativo,0,0,8


,source_id,headline,classe_finbert_original,rotulo_humano,observacao
1,134474,"Benzinga's Top Upgrades, Downgrades For August...",negativo,neutro,Manchete informativa listando upgrades e downg...
3,135142,Memo Shows Merrill Lynch to Cut Divisions to 6...,neutro,positivo,Reestruturação na subsidiária Merrill Lynch co...
6,707384,The Daily Biotech Pulse: AEterna Zentaris Expl...,negativo,neutro,Resumo diário do setor de biotecnologia com ev...
22,899722,Mid-Afternoon Market Update: Dow Slides Over 1...,positivo,neutro,Atualização de mercado com sinais mistos (qued...
25,929581,Analysts Weigh In On The Semiconductor ETF And...,negativo,neutro,Manchete genérica informando que analistas com...
29,930827,Semi Stocks Relatively Unaffected Amid Trader ...,positivo,neutro,Ações de semicondutores permanecem estáveis e ...
31,930882,Seeing Notable Block Trade In Nvidia; 1M Share...,negativo,neutro,Registro factual de negociação em bloco (block...
32,707614,Benzinga's Daily Biotech Pulse: Bellicum CFO R...,negativo,neutro,Boletim diário do setor de biotecnologia com n...
36,707549,"The Daily Biotech Pulse: Enzo's Q4 Loss, Sanof...",negativo,neutro,Resumo diário do setor de biotecnologia combin...
37,929628,"Oppenheimer Bullish On Nvidia, Cautious On AMD...",neutro,positivo,Visão otimista (bullish) da Oppenheimer especi...


## 10. Documentar e conferir a conclusão

Exportamos o resumo por empresa, o dicionário e o manifesto com hashes, modelo, parâmetros, contagens e estado da revisão. O registro descreve esta etapa de sentimento; a integração é executada separadamente no notebook 03.

In [10]:
resumo = classificadas.groupby(['stock', 'classe_finbert']).size().unstack(fill_value=0)
resumo = resumo.reindex(columns=['positivo', 'neutro', 'negativo', 'empate'], fill_value=0)
resumo['total'] = resumo.sum(axis=1)
ARQ_RESUMO = DOCS / 'resumo_sentimento_por_empresa.csv'
resumo.to_csv(ARQ_RESUMO)
definicoes = {
    'headline_hash': 'SHA-256 UTF-8 da manchete limpa enviada ao modelo.',
    'p_positivo': 'Probabilidade softmax da classe positive; não calibrada como probabilidade de acerto.',
    'p_negativo': 'Probabilidade softmax da classe negative.',
    'p_neutro': 'Probabilidade softmax da classe neutral.',
    'classe_finbert': 'Maior probabilidade por manchete; empate quando máximos diferem no máximo 1e-8.',
    'classe_diaria': 'Maior média de probabilidade por empresa/news_date; empate explícito.',
    'escore_sentimento': 'p_positivo menos p_negativo, entre -1 e 1.',
    'probabilidade_maxima': 'Maior probabilidade softmax; não é confiança calibrada.',
    'margem_top2': 'Diferença entre a maior e a segunda maior probabilidade.',
    'empate_probabilidades': 'Mais de uma probabilidade atinge o máximo na tolerância definida.',
    'n_tokens': 'Comprimento original tokenizado, incluindo tokens especiais, antes do corte.',
    'foi_truncada': 'Texto excedeu 512 tokens e foi cortado para inferência.',
    'n_noticias': 'Número de associações de notícias no grupo empresa/news_date.',
    'n_compartilhadas': 'Notícias do grupo com sinalizador de compartilhamento entre empresas.',
    'n_truncadas': 'Notícias do grupo cujo texto excedeu o limite de tokens.',
    'modelo_finbert': 'Identificador público do modelo empregado.',
    'revisao_finbert': 'Revisão imutável dos pesos e tokenizador empregados.',
}
pd.DataFrame([{'campo': k, 'definicao': v} for k, v in definicoes.items()]).to_csv(DOCS / 'dicionario_sentimento.csv', index=False)
revisao_atual = ler_revisao(ARQ_REVISAO)
ARQUIVOS_RESULTADO = [ARQ_CLASSIFICADAS, ARQ_DIARIO, ARQ_RESUMO, ARQ_REVISAO, ARQ_CACHE]
manifesto = {
    'etapa': '02_sentimento_finbert', 'inicio_utc': INICIO_EXECUCAO,
    'conclusao_utc': datetime.now(timezone.utc).isoformat(),
    'modelo': MODELO, 'revisao': REVISAO, 'configuracao_cache': config_cache, 'chave_cache': CHAVE_CACHE,
    'python': sys.version.split()[0], 'versoes': VERSOES, 'dispositivo': str(DISPOSITIVO),
    'entrada': {'arquivo': str(ENTRADA.relative_to(RAIZ)), 'sha256': HASH_ENTRADA},
    'contagens': {'associacoes': len(classificadas), 'textos_unicos': len(unicas),
                  'textos_inferidos_nesta_execucao': len(pendentes),
                  'textos_reutilizados_do_cache': len(unicas) - len(pendentes),
                  'pares_empresa_data': len(diario), 'textos_truncados': int(unicas['foi_truncada'].sum()),
                  'empates_manchetes': int(classificadas['empate_probabilidades'].sum()),
                  'empates_diarios': int(diario['empate_probabilidades'].sum()),
                  'itens_revisao': len(revisao_atual),
                  'rotulos_humanos_preenchidos': N_REVISAO_VALIDOS,
                  'revisao_comparaveis': N_COMPARAVEIS, 'revisao_concordancias': N_CONCORDANCIAS,
                  'revisao_divergencias': N_DIVERGENCIAS, 'revisao_ambiguos': N_AMBIGUOS},
    'unidade_diaria': 'stock + news_date; data civil da notícia, sem alinhamento a pregões',
    'regra_classe_diaria': 'argmax das médias das três probabilidades; empate explícito',
    'tolerancia_empate': TOL_EMPATE, 'tamanho_lote': TAMANHO_LOTE, 'semente_revisao': SEMENTE,
    'revisao_qualitativa': {'metodo': 'amostra dirigida por empresa/classe, baixa margem, compartilhamento e complemento aleatorio',
                          'revisores': sorted(set(revisao_atual['revisor'].str.strip()) - {''}),
                          'observacoes_preenchidas': int(revisao_atual['observacao'].str.strip().ne('').sum()),
                          'matriz_humano_modelo': matriz_revisao.to_dict(orient='index'),
                          'limite': 'concordancia descritiva de amostra dirigida; nao estima acuracia geral',
                          'previsoes_automaticas': 'preservadas'},
    'estado': {'inferencia': 'concluida', 'revisao_humana': ESTADO_REVISAO, 'continuidade': '03_integracao_retornos.ipynb'},
    'arquivos': {str(p.relative_to(RAIZ)): sha256_arquivo(p) for p in ARQUIVOS_RESULTADO},
}
(DOCS / 'manifesto_finbert.json').write_text(json.dumps(manifesto, indent=2, ensure_ascii=False), encoding='utf-8')
assert sha256_arquivo(ENTRADA) == HASH_ENTRADA
display(resumo)
print(f'Classificação e agregado diário exportados. Revisão humana: {ESTADO_REVISAO}. Continue no notebook 03.')

Classificação e agregado diário exportados. Revisão humana: preenchida. Continue no notebook 03.


classe_finbert,positivo,neutro,negativo,empate,total
stock,,,,,
BAC,162,524,185,0,871
JNJ,306,516,224,0,1046
KO,138,398,108,0,644
NFLX,556,1359,429,0,2344
NVDA,451,801,395,0,1647


## 11. Limitações e continuidade

A revisão indica casos em que o tom do texto inteiro difere do tom relativo a uma empresa. As divergências merecem discussão e não são automaticamente erros comprovados do modelo. Consulte o [relatório FinBERT](../docs/finbert.md) e prossiga para a [integração](03_integracao_retornos.ipynb).

**Referências:** [modelo e revisão](https://huggingface.co/ProsusAI/finbert/tree/4556d13015211d73dccd3fdd39d39232506f3e43), [artigo FinBERT](https://arxiv.org/abs/1908.10063).